# 01 - Carregamento e Preparação de Dados

Este notebook usa a configuração em YAML e os módulos do framework `model_supermarket` para carregar, normalizar e salvar a base de notas fiscais.

In [14]:
# Última nota no Nota Paraná: 02/11/2026
# Vi no dia 03/11/2026
# Última nota que apareceu foi do dia 02/11/2026

In [15]:
# !pip install -r requirements.txt

In [16]:
# IMPORTS
from pathlib import Path
import time
from src.core.config_loader import ConfigLoader
from src.core.tictoc import tictoc
from src.core.nfce_fetcher import run_nfce_fetch, process_base

# CONFIGURAÇÃO
tic_geral = time.time()
config = ConfigLoader("config/config.yaml")
qrcodes_path = Path(config.get('data.qrcodes_path'))
fetched_path = Path(config.get('data.fetched_data_path'))
mapping_path = Path(config.get('data.qrcode_mapping_path'))
raw_path = Path(config.get('data.raw_invoice_path'))
FETCH_REAL_DATA = config.get('data_fetching.enabled')
FETCH_REAL_DATA_TRAT = config.get('data_fetching.enabled_trat')
FETCH_DELAY = config.get('data_fetching.delay_between_requests')
WEATHER_ENABLED = config.get('external_data.weather.enabled')
WEATHER_OUTPUT_PATH = config.get('external_data.weather.output_path')
WEATHER_LATITUDE = config.get('external_data.weather.latitude')
WEATHER_LONGITUDE = config.get('external_data.weather.longitude')

## 1. Carregamento dos Dados

In [17]:
tic = time.time()

### 1.1 Busca no Site da Fazenda

**📦 `run_nfce_fetch`**

**📖 Descrição**

Função responsável por **buscar, extrair e estruturar dados de NFC-e a partir de QR Codes**, gerando uma base consolidada de itens de notas fiscais.

---

**⚙️ Parâmetros**

| Parâmetro       | Tipo    | Descrição                                                                |
| --------------- | ------- | ------------------------------------------------------------------------ |
| `fetch_enabled` | `bool`  | Se `True`, realiza a busca das notas. Se `False`, lê o CSV já existente. |
| `delay`         | `float` | Tempo de espera entre requisições.                                       |
| `qrcodes_path`  | `Path`  | Arquivo com os QR Codes (um por linha).                                  |
| `fetched_path`  | `Path`  | Caminho do CSV final gerado/lido.                                        |
| `mapping_path`  | `Path`  | Caminho do CSV de mapeamento da anonimização.                            |

---

**🔁 Comportamento**

**🔹 `fetch_enabled = False`**

* Lê o CSV existente (`fetched_path`)
* Retorna a base sem processamento

**🔹 `fetch_enabled = True`**

Executa as seguintes etapas:

* **Lê e remove QR Codes duplicados**
* **Consulta cada QR Code** (via API)
* **Extrai itens da nota**
* **Padroniza colunas** (schema final)
* **Converte e valida valores numéricos**
* **Enriquece com dados da nota**:

  * supermercado
  * CNPJ
  * data
  * totais
* **Anonimiza a chave da NFC-e** (SHA256)
* **Gera tabela de mapeamento (de_para)**
* **Consolida todas as notas em um único DataFrame**
* **Padroniza nome dos produtos (associar maior código ao nome mais recente)**
* **Organiza colunas finais**
* **Salva em CSV**

---

**📤 Retorno**

`pd.DataFrame` com os itens das notas fiscais processadas.

### 1.2 Execução

In [18]:
df_fetched = run_nfce_fetch(
    FETCH_REAL_DATA,
    FETCH_DELAY,
    qrcodes_path,
    fetched_path,
    mapping_path,
)

Configuração de busca (nfceget): FETCH_REAL_DATA=True
ℹ QR Codes únicos: 274
[1/274] Buscando... ✅ Nota obtida
[2/274] Buscando... ✅ Nota obtida
[3/274] Buscando... ✅ Nota obtida
[4/274] Buscando... ✅ Nota obtida
[5/274] Buscando... ✅ Nota obtida
[6/274] Buscando... ✅ Nota obtida
[7/274] Buscando... ✅ Nota obtida
[8/274] Buscando... ✅ Nota obtida
[9/274] Buscando... ✅ Nota obtida
[10/274] Buscando... ✅ Nota obtida
[11/274] Buscando... ✅ Nota obtida
[12/274] Buscando... ✅ Nota obtida
[13/274] Buscando... ✅ Nota obtida
[14/274] Buscando... ✅ Nota obtida
[15/274] Buscando... ✅ Nota obtida
[16/274] Buscando... ✅ Nota obtida
[17/274] Buscando... ✅ Nota obtida
[18/274] Buscando... ✅ Nota obtida
[19/274] Buscando... ✅ Nota obtida
[20/274] Buscando... ✅ Nota obtida
[21/274] Buscando... ✅ Nota obtida
[22/274] Buscando... ✅ Nota obtida
[23/274] Buscando... ✅ Nota obtida
[24/274] Buscando... ✅ Nota obtida
[25/274] Buscando... ✅ Nota obtida
[26/274] Buscando... ✅ Nota obtida
[27/274] Buscando... ✅

### 1.3 Amostra da Base

In [19]:
print("Nº de Linhas e Colunas:", df_fetched.shape)
df_fetched.head()

Nº de Linhas e Colunas: (4158, 13)


,CHAVE_ANONIMIZADA,DATA,CNPJ,SUPERMERCADO,QTDE_TOTAL_NOTA,VALOR_TOTAL_NOTA,VALOR_TOTAL_TRIBUTOS,COD_PRODUTO,PRODUTO,UNIDADE,QTDE,VALOR_PRODUTO,VALOR_TOTAL_PRODUTO
0,802c078fd232a7020edc6188bd3eb95e0dad9b55e711e0...,20/02/2024 18:44:16,76.430.438/0070-01,IRMAOS MUFFATO S A,5,67.53,26.84,278018,SALG ELMA CHIPS 40G,Un,1.0,3.69,3.69
1,802c078fd232a7020edc6188bd3eb95e0dad9b55e711e0...,20/02/2024 18:44:16,76.430.438/0070-01,IRMAOS MUFFATO S A,5,67.53,26.84,278018,SALG ELMA CHIPS 40G,Un,1.0,3.69,3.69
2,802c078fd232a7020edc6188bd3eb95e0dad9b55e711e0...,20/02/2024 18:44:16,76.430.438/0070-01,IRMAOS MUFFATO S A,5,67.53,26.84,264832,SALG ELMA CH 45G,Un,1.0,3.69,3.69
3,802c078fd232a7020edc6188bd3eb95e0dad9b55e711e0...,20/02/2024 18:44:16,76.430.438/0070-01,IRMAOS MUFFATO S A,5,67.53,26.84,300337,AZEITE O LIVE 250ML,Un,1.0,21.90,21.90
4,802c078fd232a7020edc6188bd3eb95e0dad9b55e711e0...,20/02/2024 18:44:16,76.430.438/0070-01,IRMAOS MUFFATO S A,5,67.53,26.84,12440,CERV SKOL 350ML,Un,12.0,2.88,34.56


In [20]:
toc = time.time()
print(f"\n\033[33mEtapa 1 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 1 | Tempo: (00:03:37) [04/11/2026 00:13:47 - 04/11/2026 00:17:24]


## 2. Preparação dos Dados

In [21]:
tic = time.time()

### 2.1 Processo de Preparação

**📦 `process_base`**

**📖 Descrição**

Função responsável por:

- processar e enriquecer a base de NFC-e
- realizar engenharia de features internas
- integrar dados externos climáticos
- padronizar produtos e unidades
- gerar dataset analítico consolidado
- ou carregar uma versão já tratada da base

dependendo do parâmetro `fetch_enabled`.

---

**⚙️ Parâmetros**

| Parâmetro | Tipo | Descrição |
|---|---|---|
| `fetch_enabled` | `bool` | Se `True`, processa os dados. Se `False`, apenas lê o `.xlsx`. |
| `df_fetched` | `pd.DataFrame` | Base bruta gerada anteriormente. |
| `raw_path` | `Path` | Caminho do arquivo final `.xlsx`. |
| `weather_enabled` | `bool` | Ativa integração climática externa. |
| `weather_output_path` | `str` | Caminho da base climática `.parquet`. |
| `latitude` | `float` | Latitude utilizada na API climática. |
| `longitude` | `float` | Longitude utilizada na API climática. |

---

**🔁 Comportamento**

**🔹 `fetch_enabled = False`**

- Lê o arquivo `.xlsx`
- Retorna a base já processada

**🔹 `fetch_enabled = True`**

Aplica as seguintes transformações:

---

**🧹 Tratamento Inicial**

**🔸 Filtragem de supermercados**

Mantém apenas:

- Condor
- Max
- Assaí

**🔸 Padronização**

- normalização de supermercados
- padronização de produtos
- padronização de unidades

---

**🕒 Feature Engineering Interno**

**🔸 Features temporais**

Criação das variáveis:

| Feature | Descrição |
|---|---|
| `MES_ANO` | Ano/mês no formato `YYYYMM` |
| `PERIODO` | MADRUGADA, MANHA, TARDE, NOITE |
| `DIA_SEMANA` | Nome do dia da semana |
| `FERIADO` | Indicador de feriado no Paraná |
| `ESTACAO_ANO` | VERAO, OUTONO, INVERNO, PRIMAVERA |

**🔸 Classificação de produtos**

Criação de:

| Feature | Descrição |
|---|---|
| `CAT_PRODUTO` | Categoria analítica do produto |

Exemplos:

- bebidas
- congelados
- bomboniere
- açougue
- hortifruti
- higiene e limpeza

**🔸 Resolução de inconsistências**

- mantém o nome mais recente por código de produto
- reduz duplicidade semântica de produtos

---

**🌦️ Integração Climática Externa**

Quando `weather_enabled=True`:

**🔸 Download automático**

Integração com:

**🌍 Open-Meteo API**

Dados históricos climáticos de Curitiba.

---

**🔸 Features climáticas**

| Feature | Descrição |
|---|---|
| `TEMPERATURA_MAX` | Temperatura máxima diária |
| `TEMPERATURA_MIN` | Temperatura mínima diária |
| `TEMPERATURA_MEDIA` | Temperatura média diária |
| `CHUVA_MM` | Volume de precipitação |
| `DIA_CHUVOSO` | Indicador booleano de chuva |
| `CAT_TEMPERATURA` | Categoria térmica do dia |

---

**🔸 Categorias de temperatura**

| Categoria |
|---|
| `MUITO_FRIO` |
| `FRIO` |
| `AMENO` |
| `QUENTE` |
| `MUITO_QUENTE` |

---

**📦 Persistência**

**🔸 Dados tratados**

Exportados em:

```text
.xlsx
```

### 2.2 Execução

In [22]:
df_trat = process_base(
    FETCH_REAL_DATA_TRAT,
    df_fetched,
    raw_path,
    WEATHER_ENABLED,
    WEATHER_OUTPUT_PATH,
    WEATHER_LATITUDE,
    WEATHER_LONGITUDE
)

Configuração de busca (nfceget): FETCH_REAL_DATA=True
✅ CNPJ filtrados: 10
✅ Supermercados normalizados: 6
✅ Datas convertidas, meses/anos únicos: 52
✅ Períodos classificados: 4
✅ Features temporais criadas
✅ Estação do ano criada
🌦 Processando dados climáticos...
🌦 Baixando dados climáticos...
✅ Dados climáticos salvos em: data\external\weather\weather_curitiba.parquet
✅ Features climáticas adicionadas
✅ Produtos padronizados: 1281
✅ Categorias classificadas: 20
✅ Unidades padronizadas: 10
✅ Nome dos produtos padronizados
✅ DataFrame final com colunas ordenadas
✅ DataFrame exportado: data\notas_fiscais_supermercado.xlsx


### 2.3 Amostra da Base

In [23]:
print("Nº de Linhas e Colunas:", df_trat.shape)
df_trat.head()

Nº de Linhas e Colunas: (4074, 25)


,chave_anonimizada,data_hora,mes_ano,dia_semana,feriado,estacao_ano,temperatura_max,temperatura_min,temperatura_media,chuva_mm,...,qtd_total_nota,valor_total_nota,valor_total_tributos,cod_produto,categoria_produto,produto,unidade,quantidade,preco_unitario,preco_total
1126,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,10166630001,MASSA,NISSIN TALH 99G BOL,UNIDADE,6.0,2.69,16.14
1186,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,11115310001,UTILIDADES,VELA PARA N3 C/6 BCA,UNIDADE,1.0,5.98,5.98
1185,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,11158780001,BEBIDA NAO ALCOOLICA,BEB MONSTER 473ML MG,UNIDADE,1.0,8.49,8.49
1184,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,11158780001,BEBIDA NAO ALCOOLICA,BEB MONSTER 473ML MG,UNIDADE,1.0,8.49,8.49
1182,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,248736,BOMBONIERE,BOMBOM LACTA 250 6G,UNIDADE,1.0,8.99,8.99


In [24]:
df_trat.columns

Index(['chave_anonimizada', 'data_hora', 'mes_ano', 'dia_semana', 'feriado',
       'estacao_ano', 'temperatura_max', 'temperatura_min',
       'temperatura_media', 'chuva_mm', 'cat_temperatura', 'dia_chuvoso',
       'periodo_dia', 'cnpj', 'supermercado', 'qtd_total_nota',
       'valor_total_nota', 'valor_total_tributos', 'cod_produto',
       'categoria_produto', 'produto', 'unidade', 'quantidade',
       'preco_unitario', 'preco_total'],
      dtype='str')

In [25]:
toc = time.time()
print(f"\n\033[33mEtapa 2 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 2 | Tempo: (00:00:04) [04/11/2026 00:17:24 - 04/11/2026 00:17:29]


## 3. Tempo Decorrido

In [26]:
toc_geral = time.time()
print(f"\n\033[33mTempo decSorrido no notebook:\033[0;0m {tictoc(tic_geral, toc_geral)}")


Tempo decSorrido no notebook: (00:03:42) [04/11/2026 00:13:47 - 04/11/2026 00:17:29]
